# Train latent-displacement visualization models

Fit and persist exactly one latent-displacement MIMIC model per configured dataset. These models are independent of experiment folds and are consumed by `04_view_latent_displacement_embeddings.ipynb`.

In [ ]:
PROFILE = "full"
EXPERIMENT_BASE_NAME = "latent-displacement-best"
SAMPLE_SIZE = 4000
MIMIC_MODE = "factorised"
CAPACITY = 0.85
N_NEIGHBORS = 14
LAMBDA_RANGE = (0.25, 1.1)
EXPERIMENT_NAME = (
    f"{EXPERIMENT_BASE_NAME}__mode-{MIMIC_MODE}"
    f"__capacity-{CAPACITY:g}__neighbors-{N_NEIGHBORS}"
    f"__lambda-{LAMBDA_RANGE[0]:g}-{LAMBDA_RANGE[1]:g}"
    f"__rows-{SAMPLE_SIZE}"
)
SEED = 0
MAX_PLOT_ROWS = 750
CONFIG_PATH = None
ARTIFACT_DIR = None

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from streamlined.config import experiment_artifact_dir, ensure_artifact_dirs, load_config
from streamlined.embedding_visualization import (
    LatentDisplacementRun,
    fit_and_save_latent_displacement_embeddings,
    save_latest_latent_displacement_run,
)

In [ ]:
config_path = Path(CONFIG_PATH) if CONFIG_PATH else EXPERIMENT_ROOT / "configs" / f"{PROFILE}.yaml"
artifact_base_dir = ARTIFACT_DIR or EXPERIMENT_ROOT / "artifacts"
artifact_dir = experiment_artifact_dir(artifact_base_dir, EXPERIMENT_NAME)
config = load_config(config_path, artifact_dir=artifact_dir)
config = replace(
    config,
    mimic_mode=MIMIC_MODE,
    mimic_capacity=CAPACITY,
    n_neighbors=N_NEIGHBORS,
    lambda_range=LAMBDA_RANGE,
)
ensure_artifact_dirs(config)

print(f"Training one model for each of: {', '.join(config.profile.datasets)}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(
    f"Sample size: {SAMPLE_SIZE}; mode: {MIMIC_MODE}; capacity: {CAPACITY}; "
    f"neighbors: {N_NEIGHBORS}; lambda range: {LAMBDA_RANGE}; seed: {SEED}"
)
artifacts = fit_and_save_latent_displacement_embeddings(
    config,
    sample_size=SAMPLE_SIZE,
    seed=SEED,
    max_plot_rows=MAX_PLOT_ROWS,
)
latest_run_path = save_latest_latent_displacement_run(
    LatentDisplacementRun(
        config=config,
        sample_size=SAMPLE_SIZE,
        seed=SEED,
        max_plot_rows=MAX_PLOT_ROWS,
    )
)
print(f"Saved latest-run manifest: {latest_run_path}")
display(artifacts)